# EEGNet LOSO — src-mirror

Thin driver that runs EEGNet LOSO using the exact code path the Hydra runner uses
(`src.backbones.eegnet.EEGNetBackbone`, `src.protocols.LOSOProtocol`,
`src.core.pipeline.Pipeline`). No reimplemented training loop, no notebook-local
preprocessing. The only knob is `N_HELD_OUT` (subject subsample) so the notebook
finishes in a reasonable time; full LOSO is `N_HELD_OUT=None`.

Mirrors `src/configs/experiment/eegnet_ra_static_loso.yaml`:
- backbone: `EEGNetBackbone` defaults from `src/configs/backbone/eegnet.yaml`
- data: 100 Hz, 0.5–40 Hz bandpass, 4 s trials
- adapter: `ra_static` (toggleable below)


In [1]:
from __future__ import annotations

import sys, time
from pathlib import Path

import numpy as np

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Side-effect imports populate the registries.
import src.backbones  # noqa: F401
import src.adapters   # noqa: F401
import src.protocols  # noqa: F401

from src.core.pipeline import Pipeline
from src.core.registry import build_from_cfg
from src.data.eegmmi import EXCLUDED_SUBJECTS, load_dataset
from src.protocols import LOSOProtocol

RAW_DIR = REPO_ROOT / "data" / "raw" / "eegmmi"
assert RAW_DIR.exists(), f"missing dataset dir: {RAW_DIR}"


In [2]:
# --- sweep config (the only knobs) ---
SEED         = 0
N_HELD_OUT   = 8       # None = full N-fold LOSO over all 104 subjects
USE_RA_STATIC = True   # mirror eegnet_ra_static_loso.yaml; flip off for plain EEGNet

# --- backbone cfg, copied verbatim from src/configs/backbone/eegnet.yaml ---
BACKBONE_CFG = dict(
    name="eegnet",
    n_channels=64,
    n_classes=4,
    f1=8, d=2, f2=16,
    kernel_len=64,
    dropout=0.5,
    train=dict(
        batch_size=64, lr=1.0e-3, weight_decay=1.0e-4,
        epochs=50, patience=8, val_frac=0.1, seed=SEED,
    ),
)

ADAPTER_CFGS = (
    [dict(name="ra_static", regularize=1.0e-5, target_space="identity",
          zero_shot_fallback="per_trial")]
    if USE_RA_STATIC else []
)

# Match src defaults: load_dataset uses dst_fs=100, tmin=0, tmax=4, EEG_BAND=(0.5,40).
ALL_SUBJECTS = tuple(s for s in range(1, 110) if s not in EXCLUDED_SUBJECTS)
print(f"subjects available: {len(ALL_SUBJECTS)}  |  RA_static={USE_RA_STATIC}  |  N_HELD_OUT={N_HELD_OUT}")


subjects available: 104  |  RA_static=True  |  N_HELD_OUT=8


In [3]:
# Load all subjects once (the protocol decides who is held out per split).
t0 = time.time()
data = load_dataset(RAW_DIR, subjects=ALL_SUBJECTS)
print(f"loaded {len(data)} subjects in {time.time()-t0:.1f}s")
print(f"  example: S001 has {len(data[1].trials)} trials, "
      f"shape={data[1].trials[0].eeg.shape}, fs={data[1].trials[0].fs}")


loaded 104 subjects in 46.5s
  example: S001 has 90 trials, shape=(64, 400), fs=100


In [4]:
protocol = LOSOProtocol(seed=SEED, n_held_out=N_HELD_OUT)
splits = list(protocol.iter_splits(data))
print(f"protocol={protocol.name}  →  {len(splits)} splits")
print("held-out subjects:", [s.meta["held_out_subject"] for s in splits])


protocol=loso  →  8 splits
held-out subjects: [2, 5, 8, 27, 32, 51, 63, 83]


In [5]:
import pandas as pd

rows = []
for i, split in enumerate(splits):
    held = split.meta["held_out_subject"]
    backbone = build_from_cfg("backbone", BACKBONE_CFG)
    adapters = [build_from_cfg("adapter", a) for a in ADAPTER_CFGS]
    pipeline = Pipeline(backbone, adapters, head=None)

    t0 = time.time()
    pipeline.fit(split.train)
    pipeline.calibrate(split.calib)
    y_proba = pipeline.predict_concat(split.eval)
    y_true = np.asarray([t.label for t in split.eval], dtype=np.int64)
    y_pred = y_proba.argmax(axis=1).astype(np.int64)
    acc = float((y_pred == y_true).mean())
    wall = time.time() - t0

    rows.append(dict(fold=i, held_out=held,
                     n_train=len(split.train), n_eval=len(split.eval),
                     acc=acc, wall_s=wall))
    print(f"  [{i+1}/{len(splits)}]  S{held:03d}  acc={acc:.3f}  "
          f"(n_train={len(split.train)}, n_eval={len(split.eval)}, {wall:.1f}s)")

df = pd.DataFrame(rows)
print()
print(f"mean accuracy = {df['acc'].mean():.3f} ± {df['acc'].std():.3f}  "
      f"(n={len(df)} folds)")
df


  [1/8]  S002  acc=0.444  (n_train=9228, n_eval=90, 221.2s)
  [2/8]  S005  acc=0.378  (n_train=9228, n_eval=90, 341.8s)


KeyboardInterrupt: 

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(df["held_out"].astype(str), df["acc"], color="C0", alpha=0.85)
ax.axhline(df["acc"].mean(), ls="--", color="C3",
           label=f"mean={df['acc'].mean():.3f}")
ax.axhline(0.25, ls=":", color="grey", label="chance (4-class)")
ax.set_xlabel("held-out subject")
ax.set_ylabel("LOSO accuracy")
tag = "EEGNet + RA-static" if USE_RA_STATIC else "EEGNet"
ax.set_title(f"{tag}  LOSO  (src-mirror, {len(df)} held-out subjects)")
ax.legend()
fig.tight_layout()
fig.show()


In [ ]:
# Optional: persist results for later comparison.
out_dir = REPO_ROOT / "results" / "eegnet_loso_src_mirror_nb"
out_dir.mkdir(parents=True, exist_ok=True)
stamp = time.strftime("%Y%m%d-%H%M%S")
tag = "ra_static" if USE_RA_STATIC else "plain"
df.to_csv(out_dir / f"loso_{tag}_{stamp}.csv", index=False)
print("wrote:", out_dir)
